# Phase 2 — Zero-shot Baseline (Groq API)

**Model:** llama-3.3-70b-versatile (zero-shot, no fine-tuning)  
**Input:** `../data/processed/test.json`  
**Output:** `../results/baseline_groq.json`  

This notebook establishes an external baseline by running zero-shot inference
on the ViMedAQA test set using the Groq API, then computing ROUGE, BLEU-4,
and BERTScore metrics for comparison against fine-tuned models in Phase 4.

> All paths are relative to `notebooks/` so this notebook runs on any machine
> without reconfiguration.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    "groq",             # Bỏ giới hạn phiên bản ==0.5.0
    "httpx<0.28.0",     # Ép hạ cấp httpx xuống phiên bản tương thích
    "evaluate==0.4.1",
    "rouge-score==0.1.2",
    "bert-score==0.3.13",
    "sacrebleu==2.3.1",
    "pandas==2.2.2",
    "tqdm==4.66.2",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed.")

## Cell 2 — Imports & Path Setup

In [ ]:
import json
import os
import time

import pandas as pd
import itertools
from groq import Groq
from tqdm import tqdm
import evaluate
import numpy as np

# Paths relative to notebooks/
TEST_DATA_PATH  = os.path.join("..", "data", "processed", "test.json")
RESULTS_DIR     = os.path.join("..", "results")
OUTPUT_PATH     = os.path.join(RESULTS_DIR, "baseline_groq.json")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Paths configured:")
print(f"  Test data : {os.path.abspath(TEST_DATA_PATH)}")
print(f"  Output    : {os.path.abspath(OUTPUT_PATH)}")

## Cell 3 — Groq API Configuration

Get your free API key from https://console.groq.com/keys 
Set it as an environment variable before running, or paste it directly into
the `GROQ_API_KEY` variable below.

In [ ]:
# Option 1 (recommended): set via environment variable before launching Jupyter
#
# Option 2: paste key directly 

print("=== MULTIPLE API KEYS CONFIGURATION ===")
print("Enter your Groq API Keys one by one (Press Enter on an empty line to finish):")

api_keys = []
for i in range(1, 10):
    key = input(f"API Key {i}: ").strip()
    if not key:
        break
    api_keys.append(key)

if not api_keys:
    raise ValueError("ERROR: You must provide at least 1 API Key!")

print(f"\n✅ Registered {len(api_keys)} API Keys. The system will rotate them automatically.")
MODEL_NAME = "llama-3.3-70b-versatile"

## Cell 4 — API Connection Test

In [ ]:
try:
    test_client = Groq(api_key=api_keys[0])
    test_response = test_client.chat.completions.create(
        messages=[{"role": "user", "content": "Respond with the single word: OK"}],
        model=MODEL_NAME,
        max_tokens=10,
    )
    print(f"API connection test passed using Key 1. Model response: {test_response.choices[0].message.content.strip()}")
except Exception as e:
    raise RuntimeError(
        f"API connection test failed: {e}\n"
        "Check your API keys and network connection."
    )

## Cell 5 — Load Test Data

In [ ]:
if not os.path.isfile(TEST_DATA_PATH):
    raise FileNotFoundError(
        f"Test file not found: {os.path.abspath(TEST_DATA_PATH)}\n"
        "Run notebook 01_data_exploration.ipynb first to generate the data splits."
    )

with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_df = pd.DataFrame(test_data)
print(f"Test samples loaded: {len(test_df)}")
print(f"Columns: {test_df.columns.tolist()}")
print("\nSample (first row):")
print(test_df.iloc[0].to_dict())

## Cell 6 — Zero-shot Inference

Rate limit note: Gemini free tier allows ~15 requests/min.  
`time.sleep(1.0)` keeps the request rate safely below that limit.

In [ ]:
PROMPT_TEMPLATE = (
    "Bạn là một chuyên gia y tế có kinh nghiệm. "
    "Dựa vào thông tin dưới đây, hãy trả lời câu hỏi một cách chính xác và ngắn gọn bằng tiếng Việt.\n\n"
    "Context: {context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

def generate_answer(question: str, context: str, current_key: str) -> str:
    """Configures the current API key and dispatches the request."""
    client = Groq(api_key=current_key)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL_NAME,
        temperature=0.0,
        max_tokens=256,
    )
    return response.choices[0].message.content.strip()

CHECKPOINT_PATH = os.path.join(RESULTS_DIR, "baseline_groq_checkpoint.jsonl")

# Use Dictionary to accurately map indices, avoiding misalignment when re-running failed samples
predictions_dict = {}

# 1. RESTORE CHECKPOINT (Only load SUCCESSFUL samples)
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line.strip())
            # Skip failed samples, forcing the system to re-run them
            if not record.get("failed", False):
                predictions_dict[record["index"]] = record["prediction"]
    print(f"Restored {len(predictions_dict)} SUCCESSFUL samples. Previous failed samples will be re-run.")

# Calculate safe delay based on the number of available keys
DELAY_SECONDS = max(2.1 / len(api_keys), 1.0) if api_keys else 2.1

# 2. INFERENCE LOOP
with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
    for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Groq zero-shot inference"):
        
        # Skip if the sample was already successfully processed
        if i in predictions_dict:
            continue
            
        success = False
        
        # Keep trying as long as there are active API keys in the list
        while not success and api_keys:
            current_key = api_keys[0]  # Always try the top key in the queue
            
            try:
                pred = generate_answer(row["question"], row["context"], current_key)
                success = True
                predictions_dict[i] = pred
                
                # Append data and flush to disk immediately
                record = {
                    "index": i,
                    "question": row["question"],
                    "reference": row["answer"],
                    "prediction": pred,
                    "failed": False
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                f.flush()
                time.sleep(DELAY_SECONDS)
                
            except Exception as err:
                err_msg = str(err)
                
                # REMOVE DEAD KEY IF RATE LIMIT IS REACHED
                if "rate limit" in err_msg.lower() or "429" in err_msg:
                    print(f"\n[WARNING] API Key ending in ...{current_key[-4:]} reached rate limit. Removing from rotation...")
                    api_keys.pop(0)  # Remove the exhausted key
                    
                    if api_keys:
                        # Recalculate delay for remaining keys
                        DELAY_SECONDS = max(2.1 / len(api_keys), 1.0)
                        print(f"Switching to the next key. {len(api_keys)} key(s) remaining.")
                    else:
                        print("\n[CRITICAL ERROR] ALL API KEYS HAVE EXHAUSTED THEIR RATE LIMITS!")
                        break
                else:
                    # Standard execution error (e.g., parsing, 503 server error)
                    print(f"\n[WARNING] Sample {i} failed with standard error: {err_msg}")
                    break  # Break out of the while loop to mark this specific sample as failed
                    
        # If execution exits the while loop without success, log it as a failure
        if not success:
            record = {
                "index": i,
                "question": row["question"],
                "reference": row["answer"],
                "prediction": "",
                "failed": True
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()
            
        # Stop entirely if all keys are dead
        if not api_keys:
            break

# 3. RECONSTRUCT LISTS FOR DOWNSTREAM EVALUATION CELLS (Cell 7, 8)
predictions = []
references = []
failed_indices = []

for i, row in test_df.iterrows():
    if i in predictions_dict:
        predictions.append(predictions_dict[i])
        references.append(row["answer"])
    else:
        predictions.append("")
        references.append(row["answer"])
        failed_indices.append(i)

print(f"\nCurrent progress: Completed {len(predictions_dict)}/{len(test_df)} samples.")

## Cell 7 — Compute Metrics

Empty predictions (from API errors) are excluded before scoring.

In [ ]:
# Ép kiểu numpy.int64 sang int nguyên bản để tương thích với JSON
clean_failed_indices = [int(idx) for idx in failed_indices]

# Build a detailed record including metadata, scores, and per-sample predictions
baseline_results = {
    "model":         MODEL_NAME,
    "mode":          "zero-shot",
    "num_total":     len(predictions),
    "num_valid":     len(preds_valid),
    "num_failed":    len(clean_failed_indices),
    "failed_indices": clean_failed_indices,
    "rouge1":        round(rouge_scores["rouge1"], 4),
    "rouge2":        round(rouge_scores["rouge2"], 4),
    "rougeL":        round(rouge_scores["rougeL"], 4),
    "bleu4":         bleu4,
    "bertscore_f1":  bertscore_f1,
    "predictions": [
        {
            "index":      int(test_df.index[i]),
            "topic":      test_df.iloc[i]["topic"],
            "question":   test_df.iloc[i]["question"],
            "reference":  references[i],
            "prediction": predictions[i],
            "failed":     (int(test_df.index[i]) in clean_failed_indices),
        }
        for i in range(len(predictions))
    ],
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print(f"Results saved -> {os.path.abspath(OUTPUT_PATH)}")
print("\nSummary:")
print(f"  Model        : {baseline_results['model']}")
print(f"  Mode         : {baseline_results['mode']}")
print(f"  Valid samples: {baseline_results['num_valid']} / {baseline_results['num_total']}")
print(f"  ROUGE-1      : {baseline_results['rouge1']}")
print(f"  ROUGE-2      : {baseline_results['rouge2']}")
print(f"  ROUGE-L      : {baseline_results['rougeL']}")
print(f"  BLEU-4       : {baseline_results['bleu4']}")
print(f"  BERTScore F1 : {baseline_results['bertscore_f1']}")

## Cell 8 — Save Results

In [ ]:
# Build a detailed record including metadata, scores, and per-sample predictions
baseline_results = {
    "model":         MODEL_NAME,
    "mode":          "zero-shot",
    "num_total":     len(predictions),
    "num_valid":     len(preds_valid),
    "num_failed":    len(failed_indices),
    "failed_indices": failed_indices,
    "rouge1":        round(rouge_scores["rouge1"], 4),
    "rouge2":        round(rouge_scores["rouge2"], 4),
    "rougeL":        round(rouge_scores["rougeL"], 4),
    "bleu4":         bleu4,
    "bertscore_f1":  bertscore_f1,
    "predictions": [
        {
            "index":      int(test_df.index[i]),
            "topic":      test_df.iloc[i]["topic"],
            "question":   test_df.iloc[i]["question"],
            "reference":  references[i],
            "prediction": predictions[i],
            "failed":     (i in failed_indices),
        }
        for i in range(len(predictions))
    ],
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print(f"Results saved -> {os.path.abspath(OUTPUT_PATH)}")
print("\nSummary:")
print(f"  Model        : {baseline_results['model']}")
print(f"  Mode         : {baseline_results['mode']}")
print(f"  Valid samples: {baseline_results['num_valid']} / {baseline_results['num_total']}")
print(f"  ROUGE-1      : {baseline_results['rouge1']}")
print(f"  ROUGE-2      : {baseline_results['rouge2']}")
print(f"  ROUGE-L      : {baseline_results['rougeL']}")
print(f"  BLEU-4       : {baseline_results['bleu4']}")
print(f"  BERTScore F1 : {baseline_results['bertscore_f1']}")

## Cell 9 — Phase 2 Checklist Verification

In [ ]:
expected_files = [
    OUTPUT_PATH,
]

print("Phase 2 Output Verification:")
all_ok = True
for fp in expected_files:
    abs_fp = os.path.abspath(fp)
    exists = os.path.isfile(abs_fp)
    size   = os.path.getsize(abs_fp) if exists else 0
    status = "OK" if (exists and size > 0) else "MISSING"
    if status != "OK":
        all_ok = False
    print(f"  [{status}] {os.path.basename(abs_fp)} ({size:,} bytes)")

if all_ok:
    print(
        "\nAll Phase 2 outputs verified.\n"
        "Note down ROUGE-L and BLEU-4 scores above — these are the zero-shot "
        "reference scores for comparison in Phase 4 (unified evaluation)."
    )
else:
    print("\nSome outputs are missing. Re-run the cells above.")